In [0]:
spark.conf.set(
"fs.azure.account.key.<STORAGEACCOUNT>.blob.core.windows.net",
"Access key"
)
display(dbutils.fs.ls("wasbs://retail@<STORAGEACCOUNT>.blob.core.windows.net/"))


path,name,size,modificationTime
wasbs://retail@pocstorageretail.blob.core.windows.net/bronze/,bronze/,0,0
wasbs://retail@pocstorageretail.blob.core.windows.net/gold/,gold/,0,0
wasbs://retail@pocstorageretail.blob.core.windows.net/silver/,silver/,0,0


In [0]:
display(dbutils.fs.ls("wasbs://retail@pocstorageretail.blob.core.windows.net/bronze/"))

path,name,size,modificationTime
wasbs://retail@pocstorageretail.blob.core.windows.net/bronze/customer/,customer/,0,0
wasbs://retail@pocstorageretail.blob.core.windows.net/bronze/product/,product/,0,0
wasbs://retail@pocstorageretail.blob.core.windows.net/bronze/store/,store/,0,0
wasbs://retail@pocstorageretail.blob.core.windows.net/bronze/transaction/,transaction/,0,0


In [0]:
display(dbutils.fs.ls("wasbs://retail@pocstorageretail.blob.core.windows.net/bronze/transaction"))

path,name,size,modificationTime
wasbs://retail@pocstorageretail.blob.core.windows.net/bronze/transaction/dbo.transactions.parquet,dbo.transactions.parquet,1757,1773420475000


In [0]:
df_transactions=spark.read.parquet("wasbs://retail@pocstorageretail.blob.core.windows.net/bronze/transaction")
df_products=spark.read.parquet("wasbs://retail@pocstorageretail.blob.core.windows.net/bronze/product")
df_stores=spark.read.parquet("wasbs://retail@pocstorageretail.blob.core.windows.net/bronze/store")
df_customers=spark.read.parquet("wasbs://retail@pocstorageretail.blob.core.windows.net/bronze/customer/chethan327/End-to-End-Azure-Data-Engineering-PROJECT-ADF-DATABRICKS-POWER-BI/refs/heads/main/customers.parquet")
display(df_customers)

customer_id,first_name,last_name,email,phone,city,registration_date
101,Ravi,Yadav,user101@example.com,9887654321,Delhi,2023-09-14
102,Nina,Joshi,user102@example.com,9876543210,Mumbai,2024-01-21
103,Sonal,Sharma,user103@example.com,9865432109,Bangalore,2023-07-10
104,Karan,Patel,user104@example.com,9854321098,Hyderabad,2024-02-05
105,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28
106,Ajay,Mishra,user106@example.com,9832109876,Pune,2024-03-10
107,Priya,Kapoor,user107@example.com,9821098765,Ahmedabad,2023-05-12
108,Rahul,Verma,user108@example.com,9810987654,Kolkata,2023-08-19
109,Pooja,Mehta,user109@example.com,9809876543,Delhi,2024-04-01
110,Deepak,Nair,user110@example.com,9798765432,Mumbai,2023-10-14


In [0]:
# COMMAND ----------

# DBTITLE 1,create silver layer - data cleaning
from pyspark.sql.functions import col

# Convert types and clean data
df_transactions = df_transactions.select(
    col("transaction_id").cast("int"),
    col("customer_id").cast("int"),
    col("product_id").cast("int"),
    col("store_id").cast("int"),
    col("quantity").cast("int"),
    col("transaction_date").cast("date")
)

df_products = df_products.select(
    col("product_id").cast("int"),
    col("product_name"),
    col("category"),
    col("price").cast("double")
)

df_stores = df_stores.select(
    col("store_id").cast("int"),
    col("store_name"),
    col("location")
)

df_customers = df_customers.select(
    "customer_id", "first_name", "last_name", "email", "city", "registration_date"
).dropDuplicates(["customer_id"])


In [0]:
# COMMAND ----------

# DBTITLE 1,join all data together
# Join all data
df_silver = df_transactions \
    .join(df_customers, "customer_id") \
    .join(df_products, "product_id") \
    .join(df_stores, "store_id") \
    .withColumn("total_amount", col("quantity") * col("price"))

In [0]:
display(df_silver)

store_id,product_id,customer_id,transaction_id,quantity,transaction_date,first_name,last_name,email,city,registration_date,product_name,category,price,store_name,location,total_amount
1,5,130,20,2,2024-07-30,Tanvi,Kapoor,user130@example.com,Pune,2023-12-22,Notebook Set,Stationery,149.0,City Mall Store,Mumbai,298.0
2,6,126,26,2,2024-09-21,Bhavna,Nair,user126@example.com,Mumbai,2023-07-17,Water Bottle,Fitness,299.0,High Street Store,Delhi,598.0
2,7,110,7,5,2025-01-01,Deepak,Nair,user110@example.com,Mumbai,2023-10-14,Smartwatch,Electronics,4999.0,High Street Store,Delhi,24995.0
4,9,119,29,2,2025-06-03,Kunal,Nair,user119@example.com,Bangalore,2023-08-10,Dumbbell Set,Fitness,1999.0,Downtown Mini Store,Pune,3998.0
4,8,116,30,5,2025-03-16,Rakesh,Kapoor,user116@example.com,Kolkata,2023-06-15,Desk Organizer,Accessories,399.0,Downtown Mini Store,Pune,1995.0
2,2,124,10,5,2024-08-27,Kavita,Sharma,user124@example.com,Kolkata,2023-11-15,Bluetooth Speaker,Electronics,1299.49,High Street Store,Delhi,6497.45
3,9,107,22,4,2024-11-16,Priya,Kapoor,user107@example.com,Ahmedabad,2023-05-12,Dumbbell Set,Fitness,1999.0,Tech World Outlet,Bangalore,7996.0
4,1,103,18,3,2024-09-05,Sonal,Sharma,user103@example.com,Bangalore,2023-07-10,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,2399.9700000000003
3,3,104,13,4,2025-05-04,Karan,Patel,user104@example.com,Hyderabad,2024-02-05,Yoga Mat,Fitness,499.0,Tech World Outlet,Bangalore,1996.0
4,1,120,14,5,2024-07-17,Alka,Mishra,user120@example.com,Hyderabad,2023-12-01,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,3999.95


In [0]:
silver_path = "wasbs://retail@pocstorageretail.blob.core.windows.net/silver/"

df_silver.write.mode("overwrite").format("delta").save(silver_path)

In [0]:
# DBTITLE 1,create silver dataset
spark.sql("""
CREATE TABLE IF NOT EXISTS retail_silver_cleaned
AS SELECT * FROM delta.`wasbs://retail@pocstorageretail.blob.core.windows.net/silver/`
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql select* from retail_silver_cleaned

store_id,product_id,customer_id,transaction_id,quantity,transaction_date,first_name,last_name,email,city,registration_date,product_name,category,price,store_name,location,total_amount
1,5,130,20,2,2024-07-30,Tanvi,Kapoor,user130@example.com,Pune,2023-12-22,Notebook Set,Stationery,149.0,City Mall Store,Mumbai,298.0
2,6,126,26,2,2024-09-21,Bhavna,Nair,user126@example.com,Mumbai,2023-07-17,Water Bottle,Fitness,299.0,High Street Store,Delhi,598.0
2,7,110,7,5,2025-01-01,Deepak,Nair,user110@example.com,Mumbai,2023-10-14,Smartwatch,Electronics,4999.0,High Street Store,Delhi,24995.0
4,9,119,29,2,2025-06-03,Kunal,Nair,user119@example.com,Bangalore,2023-08-10,Dumbbell Set,Fitness,1999.0,Downtown Mini Store,Pune,3998.0
4,8,116,30,5,2025-03-16,Rakesh,Kapoor,user116@example.com,Kolkata,2023-06-15,Desk Organizer,Accessories,399.0,Downtown Mini Store,Pune,1995.0
2,2,124,10,5,2024-08-27,Kavita,Sharma,user124@example.com,Kolkata,2023-11-15,Bluetooth Speaker,Electronics,1299.49,High Street Store,Delhi,6497.45
3,9,107,22,4,2024-11-16,Priya,Kapoor,user107@example.com,Ahmedabad,2023-05-12,Dumbbell Set,Fitness,1999.0,Tech World Outlet,Bangalore,7996.0
4,1,103,18,3,2024-09-05,Sonal,Sharma,user103@example.com,Bangalore,2023-07-10,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,2399.9700000000003
3,3,104,13,4,2025-05-04,Karan,Patel,user104@example.com,Hyderabad,2024-02-05,Yoga Mat,Fitness,499.0,Tech World Outlet,Bangalore,1996.0
4,1,120,14,5,2024-07-17,Alka,Mishra,user120@example.com,Hyderabad,2023-12-01,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,3999.95


In [0]:
# DBTITLE 1,gold layer
# Load cleaned transactions from Silver layer
silver_df = spark.read.format("delta").load("wasbs://retail@pocstorageretail.blob.core.windows.net/silver/")


In [0]:
display(silver_df)

store_id,product_id,customer_id,transaction_id,quantity,transaction_date,first_name,last_name,email,city,registration_date,product_name,category,price,store_name,location,total_amount
1,5,130,20,2,2024-07-30,Tanvi,Kapoor,user130@example.com,Pune,2023-12-22,Notebook Set,Stationery,149.0,City Mall Store,Mumbai,298.0
2,6,126,26,2,2024-09-21,Bhavna,Nair,user126@example.com,Mumbai,2023-07-17,Water Bottle,Fitness,299.0,High Street Store,Delhi,598.0
2,7,110,7,5,2025-01-01,Deepak,Nair,user110@example.com,Mumbai,2023-10-14,Smartwatch,Electronics,4999.0,High Street Store,Delhi,24995.0
4,9,119,29,2,2025-06-03,Kunal,Nair,user119@example.com,Bangalore,2023-08-10,Dumbbell Set,Fitness,1999.0,Downtown Mini Store,Pune,3998.0
4,8,116,30,5,2025-03-16,Rakesh,Kapoor,user116@example.com,Kolkata,2023-06-15,Desk Organizer,Accessories,399.0,Downtown Mini Store,Pune,1995.0
2,2,124,10,5,2024-08-27,Kavita,Sharma,user124@example.com,Kolkata,2023-11-15,Bluetooth Speaker,Electronics,1299.49,High Street Store,Delhi,6497.45
3,9,107,22,4,2024-11-16,Priya,Kapoor,user107@example.com,Ahmedabad,2023-05-12,Dumbbell Set,Fitness,1999.0,Tech World Outlet,Bangalore,7996.0
4,1,103,18,3,2024-09-05,Sonal,Sharma,user103@example.com,Bangalore,2023-07-10,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,2399.9700000000003
3,3,104,13,4,2025-05-04,Karan,Patel,user104@example.com,Hyderabad,2024-02-05,Yoga Mat,Fitness,499.0,Tech World Outlet,Bangalore,1996.0
4,1,120,14,5,2024-07-17,Alka,Mishra,user120@example.com,Hyderabad,2023-12-01,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,3999.95


In [0]:
from pyspark.sql.functions import sum, countDistinct, avg

gold_df = silver_df.groupBy(
    "transaction_date",
    "product_id", "product_name", "category",
    "store_id", "store_name", "location"
).agg(
    sum("quantity").alias("total_quantity_sold"),
    sum("total_amount").alias("total_sales_amount"),
    countDistinct("transaction_id").alias("number_of_transactions"),
    avg("total_amount").alias("average_transaction_value")
)


In [0]:
display(gold_df)


transaction_date,product_id,product_name,category,store_id,store_name,location,total_quantity_sold,total_sales_amount,number_of_transactions,average_transaction_value
2024-11-02,8,Desk Organizer,Accessories,1,City Mall Store,Mumbai,1,399.0,1,399.0
2024-08-11,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98
2024-12-13,8,Desk Organizer,Accessories,4,Downtown Mini Store,Pune,5,1995.0,1,1995.0
2025-05-04,3,Yoga Mat,Fitness,3,Tech World Outlet,Bangalore,4,1996.0,1,1996.0
2025-05-26,5,Notebook Set,Stationery,1,City Mall Store,Mumbai,4,596.0,1,596.0
2024-07-14,1,Wireless Mouse,Electronics,5,Mega Plaza,Chennai,1,799.99,1,799.99
2024-07-17,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,5,3999.95,1,3999.95
2024-09-05,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,3,2399.9700000000003,1,2399.9700000000003
2025-06-03,9,Dumbbell Set,Fitness,4,Downtown Mini Store,Pune,2,3998.0,1,3998.0
2024-08-27,2,Bluetooth Speaker,Electronics,2,High Street Store,Delhi,5,6497.45,1,6497.45


In [0]:
gold_path = "wasbs://retail@pocstorageretail.blob.core.windows.net/gold/"

gold_df.write.mode("overwrite").format("delta").save(gold_path)



In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS retail_gold_sales_summary
AS SELECT * FROM delta.`wasbs://retail@pocstorageretail.blob.core.windows.net/gold/`
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql select * from retail_silver_cleaned

store_id,product_id,customer_id,transaction_id,quantity,transaction_date,first_name,last_name,email,city,registration_date,product_name,category,price,store_name,location,total_amount
1,5,130,20,2,2024-07-30,Tanvi,Kapoor,user130@example.com,Pune,2023-12-22,Notebook Set,Stationery,149.0,City Mall Store,Mumbai,298.0
2,6,126,26,2,2024-09-21,Bhavna,Nair,user126@example.com,Mumbai,2023-07-17,Water Bottle,Fitness,299.0,High Street Store,Delhi,598.0
2,7,110,7,5,2025-01-01,Deepak,Nair,user110@example.com,Mumbai,2023-10-14,Smartwatch,Electronics,4999.0,High Street Store,Delhi,24995.0
4,9,119,29,2,2025-06-03,Kunal,Nair,user119@example.com,Bangalore,2023-08-10,Dumbbell Set,Fitness,1999.0,Downtown Mini Store,Pune,3998.0
4,8,116,30,5,2025-03-16,Rakesh,Kapoor,user116@example.com,Kolkata,2023-06-15,Desk Organizer,Accessories,399.0,Downtown Mini Store,Pune,1995.0
2,2,124,10,5,2024-08-27,Kavita,Sharma,user124@example.com,Kolkata,2023-11-15,Bluetooth Speaker,Electronics,1299.49,High Street Store,Delhi,6497.45
3,9,107,22,4,2024-11-16,Priya,Kapoor,user107@example.com,Ahmedabad,2023-05-12,Dumbbell Set,Fitness,1999.0,Tech World Outlet,Bangalore,7996.0
4,1,103,18,3,2024-09-05,Sonal,Sharma,user103@example.com,Bangalore,2023-07-10,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,2399.9700000000003
3,3,104,13,4,2025-05-04,Karan,Patel,user104@example.com,Hyderabad,2024-02-05,Yoga Mat,Fitness,499.0,Tech World Outlet,Bangalore,1996.0
4,1,120,14,5,2024-07-17,Alka,Mishra,user120@example.com,Hyderabad,2023-12-01,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,3999.95
